# Geração de Tabela LaTeX a partir de arquivos .txt

Este notebook lê todos os arquivos `.txt` da pasta `input`, processa os dados e gera uma tabela em formato LaTeX para ser utilizada no relatório.

## 1. Importar bibliotecas necessárias
Vamos importar as bibliotecas pandas e os para manipulação de arquivos e dados.

In [7]:
import pandas as pd
import os
from pathlib import Path

## 2. Estrutura de pastas

In [8]:
from pathlib import Path
import shutil

# Caminhos absolutos (ajuste conforme necessário)
src_base = Path.cwd().parent / 'src' / 'dados'
dst_base = Path.cwd().parent / 'relatorioCefet' / 'tabelas'

# Remove a pasta de destino inteira se existir (limpeza total)
if dst_base.exists() and dst_base.is_dir():
    shutil.rmtree(dst_base)
    print(f'Pasta de destino removida: {dst_base}')

# Cria input e output novamente
(dst_base / 'input').mkdir(parents=True, exist_ok=True)
(dst_base / 'output').mkdir(parents=True, exist_ok=True)

# Cria as subpastas de output conforme existem em src/dados/output
output_src = src_base / 'output'
if output_src.exists():
    for subpasta in output_src.iterdir():
        if subpasta.is_dir():
            (dst_base / 'output' / subpasta.name).mkdir(parents=True, exist_ok=True)
print('Estrutura de pastas criada em:', dst_base)


Pasta de destino removida: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas
Estrutura de pastas criada em: d:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas


## 2. Listar arquivos .csv na pasta input
Vamos listar todos os arquivos `.csv` presentes na pasta `input`.

In [9]:
from pathlib import Path

# Define os caminhos das pastas
input_dir = Path('../src/dados/input').resolve()
output_dir = Path('../src/dados/output').resolve()

# Lista todos os arquivos em input (não recursivo)
arquivos_input = [arq for arq in input_dir.glob('*') if arq.is_file()]

# Lista todos os arquivos em output (recursivo, incluindo subpastas)
arquivos_output = [arq for arq in output_dir.rglob('*') if arq.is_file()]

print(f'Arquivos em input ({len(arquivos_input)}):')
for arq in arquivos_input:
    print(arq)

print(f'\nArquivos em output ({len(arquivos_output)}):')
for arq in arquivos_output:
    print(arq)

Arquivos em input (16):
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Caotico_regra_30_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Caotico_regra_45_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Complexo_regra_110_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Complexo_regra_54_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Homogeneo_regra_0_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\dados\input\Homogeneo_regra_32_tamanho_101_passos_100.csv
D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWo

## 3. Ler e processar arquivos .csv
Vamos ler o conteúdo de cada arquivo `.csv` e armazenar os dados em uma lista.

In [10]:
import pandas as pd
import ast
from pathlib import Path

# ============================================================
# Funcoes auxiliares
# ============================================================

def escapar_latex(texto):
    """Escapa caracteres especiais para LaTeX."""
    return (str(texto)
            .replace('\\', r'\textbackslash ')
            .replace('_', r'\_')
            .replace('%', r'\%')
            .replace('&', r'\&')
            .replace('#', r'\#')
            .replace('{', r'\{')
            .replace('}', r'\}')
            .replace('^', r'\^{}')
            .replace('~', r'\~{}')
           )

MAPA_LATEX = {
    'alpha_est': r'$\hat{\alpha}$',
    'C_est': r'$\hat{C}$',
    'r2_ajuste': r'$R^2_{\mathrm{aj}}$',
    'std_err_alpha': r'$SE(\hat{\alpha})$',
    't_stat': r'$t$',
    't_critico': r'$t_{\mathrm{crit}}$',
    'p_valor': r'$p$-valor',
    'graus_liberdade': r'$gl$',
    'media_Z': r'$\bar{Z}$',
    'desvio_padrao_Z': r'$\sigma_Z$',
    'ks_stat': r'$D_{\mathrm{KS}}$',
    'ks_pvalor': r'$p$-valor (KS)',
    'parametro': 'parâmetro',
    'valor': 'valor',
    'N_TCL': r'$N_{\mathrm{TCL}}$',
    'N_STEPS_TCL': r'$N_{\mathrm{steps,TCL}}$',
    'N_WALKS': r'$N_{\mathrm{walks}}$',
    'N_STEPS': r'$N_{\mathrm{steps}}$',
    'LCG_A': r'$a$',
    'LCG_C': r'$c$',
    'LCG_M': r'$m$',
    'seed_formula': r'$\mathrm{seed}_i=(i+1)\cdot 9973$',
    'deslocamento_final': r'$S_N$',
    'z_padronizado': r'$Z=S_N/\sqrt{N}$',
}

def formatar_nome_variavel(nome):
    texto = str(nome)
    if texto in MAPA_LATEX:
        return MAPA_LATEX[texto]
    return escapar_latex(texto)


def para_notacao_cientifica_latex(numero):
    """Converte float para notacao cientifica em LaTeX com duas casas."""
    mantissa_str, expoente_str = f"{numero:.2e}".split("e")
    expoente = int(expoente_str)
    if expoente == 0:
        return f"{numero:.2f}".rstrip("0").rstrip(".")
    return f"$\\displaystyle {mantissa_str} \\times 10^{{{expoente}}}$"


def formatar_numero(valor):
    """Inteiros sem casas; flutuantes em notacao cientifica LaTeX com duas casas."""
    try:
        if isinstance(valor, str):
            texto = valor.strip()
            if texto.startswith('$') and texto.endswith('$'):
                return texto
            if '\\' in texto:
                return texto

        if isinstance(valor, bool):
            return escapar_latex(valor)

        if isinstance(valor, int):
            return str(valor)

        if isinstance(valor, float):
            if valor.is_integer():
                return str(int(valor))
            return para_notacao_cientifica_latex(valor)

        if isinstance(valor, str):
            texto = valor.strip()
            numero = float(texto)
            if numero.is_integer():
                return str(int(numero))
            return para_notacao_cientifica_latex(numero)
    except Exception:
        pass

    return escapar_latex(valor)


def gerar_caption(nome_base):
    nome = nome_base.replace('_', ' ').replace('-', ' ')
    nome = nome.capitalize()
    return f"Tabela - Dados referentes a {nome}."


def gerar_label(nome_base):
    nome = nome_base.replace('.', '_').replace('-', '_')
    return f"tab:{nome}"


# ============================================================
# Formatacao automatica de listas (dt_list e outras)
# ============================================================

def formatar_lista(valor):
    """
    Detecta automaticamente listas no CSV e formata para LaTeX.
    Funciona para listas longas, numeros ou strings.
    """
    try:
        lista = ast.literal_eval(str(valor))
        if not isinstance(lista, list):
            return formatar_numero(valor)
    except Exception:
        return formatar_numero(valor)

    lista = [formatar_numero(v) for v in lista]

    linhas = []
    for i in range(0, len(lista), 3):
        linhas.append(", ".join(lista[i:i+3]))

    conteudo = r" \\ ".join(linhas)

    return (
        r"\scriptsize{\begin{tabular}[c]{l}"
        + conteudo +
        r"\end{tabular}}"
    )


# ============================================================
# Funcao principal - gera tabela LaTeX
# ============================================================

def salvar_tabela_latex(arquivo_csv, pasta_saida):
    try:
        df = pd.read_csv(arquivo_csv)

        df_latex = df.copy()
        df_latex.columns = [formatar_nome_variavel(col) for col in df.columns]

        if 'parametro' in df.columns:
            nome_col_param = formatar_nome_variavel('parametro')
            df_latex[nome_col_param] = df['parametro'].apply(formatar_nome_variavel)

        # Formatacao automatica
        for col in df_latex.columns:
            df_latex[col] = df_latex[col].apply(formatar_lista)

        nome_base = arquivo_csv.stem
        caption = gerar_caption(nome_base)
        label = gerar_label(nome_base)

        # Detecta colunas que sao listas
        colunas_lista = []
        for col in df.columns:
            try:
                if df[col].astype(str).str.startswith("[").any():
                    colunas_lista.append(col)
            except Exception:
                pass

        # Ajuste automatico da largura da ultima coluna se houver listas
        if colunas_lista:
            col_format = " ".join(
                ["l"] * (len(df_latex.columns) - 1) + ["p{8cm}"]
            )
        else:
            col_format = None

        tabela = df_latex.to_latex(
            index=False,
            escape=False,
            column_format=col_format
        ) if col_format else df_latex.to_latex(index=False, escape=False)

        conteudo = (
            "\\begin{table}[H]\n"
            "\\centering\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            f"{tabela}\n"
            "\\end{table}\n"
        )

        nome_saida = arquivo_csv.with_suffix(".tex").name
        caminho_saida = pasta_saida / nome_saida
        pasta_saida.mkdir(parents=True, exist_ok=True)

        with open(caminho_saida, "w", encoding="utf-8") as f:
            f.write(conteudo)

        print(f"Tabela LaTeX salva em: {caminho_saida}")

    except Exception as e:
        print(f"Erro ao processar {arquivo_csv}: {e}")


# ============================================================
# Processamento dos arquivos de INPUT
# ============================================================

pasta_saida_input = Path('../relatorioCefet/tabelas/input').resolve()

if not arquivos_input:
    print("Nenhum arquivo encontrado em input.")
else:
    for arquivo in arquivos_input:
        salvar_tabela_latex(arquivo, pasta_saida_input)


# ============================================================
# Processamento dos arquivos de OUTPUT (mantendo subpastas)
# ============================================================

pasta_saida_output = Path('../relatorioCefet/tabelas/output').resolve()
output_dir = Path('../src/dados/output').resolve()

if not arquivos_output:
    print("Nenhum arquivo encontrado em output.")
else:
    for arquivo in arquivos_output:
        subpath = arquivo.relative_to(output_dir).parent
        destino = pasta_saida_output / subpath
        destino.mkdir(parents=True, exist_ok=True)
        salvar_tabela_latex(arquivo, destino)

print("Processo finalizado.")

Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas\input\Caotico_regra_30_tamanho_101_passos_100.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas\input\Caotico_regra_45_tamanho_101_passos_100.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas\input\Complexo_regra_110_tamanho_101_passos_100.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas\input\Complexo_regra_54_tamanho_101_passos_100.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\tabelas\input\Homogeneo_regra_0_tamanho_101_passos_100.tex
Tabela LaTeX salva em: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica

## 8. Copiar toda a pasta de gráficos para figuras

Esta célula copia recursivamente todo o conteúdo da pasta `graficos` (incluindo subpastas e arquivos) para a pasta `relatorioCefet/figuras`. Útil para garantir que todos os gráficos estejam disponíveis no relatório.

In [11]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    # Remove a pasta de destino se já existir
    if destino.exists() and destino.is_dir():
        shutil.rmtree(destino)
        print(f'Pasta de destino removida: {destino}')
    # Cria a pasta de destino novamente
    destino.mkdir(parents=True, exist_ok=True)
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)


Pasta de destino removida: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Caotico_regra_30_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos\Caotico_regra_30_tamanho_101_passos_100.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Caotico_regra_45_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos\Caotico_regra_45_tamanho_101_passos_100.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Complexo_regra_110_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\Pri

In [12]:
import shutil
from pathlib import Path

# Caminhos de origem e destino
graficos_dir = Path('../src/graficos').resolve()
figuras_dir = Path('../relatorioCefet/figuras').resolve()
destino_graficos = figuras_dir / 'graficos'

def copiar_pasta(origem, destino):
    if not origem.exists():
        print(f'Pasta de origem não existe: {origem}')
        return
    for item in origem.rglob('*'):
        if item.is_file():
            destino_arquivo = destino / item.relative_to(origem)
            destino_arquivo.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, destino_arquivo)
            print(f'Arquivo copiado: {item} -> {destino_arquivo}')

copiar_pasta(graficos_dir, destino_graficos)



Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Caotico_regra_30_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos\Caotico_regra_30_tamanho_101_passos_100.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Caotico_regra_45_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos\Caotico_regra_45_tamanho_101_passos_100.png
Arquivo copiado: D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\src\graficos\Complexo_regra_110_tamanho_101_passos_100.png -> D:\GitHub\DoutoradoCefet\PrincipioModelagemMatematica\Trabalhos\PraticaAutomatosWolfram\relatorioCefet\figuras\graficos\Complexo_regra_110_tamanho_101_passos_100.png
Arquivo c

---
## Referências

- Slides `pmmat_aula07.pdf`: parâmetros do LCG e função `aleat()`.
- Slides `numerosaleatorios.pdf`: geradores congruenciais e aplicações.
- Random Walk 1D: distribuição binomial e aproximação gaussiana (TCL).